In [11]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils import resample
from pathlib import Path
np.random.seed(42)

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
INPUT_FILE = PROJECT_ROOT / "data" / "mlp_data.csv"
TRAIN_FILE = PROJECT_ROOT / "outputs" / "mlp_train_preprocessed.csv"
TEST_FILE = PROJECT_ROOT / "outputs" / "mlp_test_preprocessed.csv"

In [13]:
df = pd.read_csv(INPUT_FILE)
df.head()

,zip_code,date,MORTGAGE30US,UNRATE,CPIAUCSL,zhvi_yoy_pct,inventory,days_to_pending,tier_boston_city,tier_inner_suburb,tier_outer_suburb,high_stress,split
0,1826,2021-01-31,-1.315871,0.857990,-0.692296,1.025253,-0.753537,0.652581,0,0,1,0,train
1,2333,2020-04-30,-0.949574,4.929049,-0.975248,-0.543465,-0.159508,0.208728,0,0,1,0,train
2,2459,2018-05-31,-0.128451,-0.402099,-1.198038,-0.164461,0.774547,-0.568015,0,0,1,0,train
3,3053,2020-04-30,-0.949574,4.929049,-0.975248,0.108575,-0.159508,0.208728,0,0,1,0,train
4,2138,2023-12-31,1.301457,-0.402099,1.265793,-1.118338,-1.169087,-0.013199,0,1,0,1,train


In [14]:
# Extract y variable
y = df['high_stress'].values
y = y.reshape(-1, 1)

# Split data into train and test sets
y_train = y[df['split'] == 'train']
y_test = y[df['split'] == 'test']

y[:5], y_train[:5], y_test[:5]

(array([[0],
        [0],
        [0],
        [0],
        [1]]),
 array([[0],
        [0],
        [0],
        [0],
        [1]]),
 array([[0],
        [0],
        [0],
        [0],
        [1]]))

In [15]:
# Extract date features
df['day'] = pd.to_datetime(df['date']).dt.day
df['month'] = pd.to_datetime(df['date']).dt.month
df['year'] = pd.to_datetime(df['date']).dt.year

# One-hot encode zip_code
df = pd.get_dummies(df, columns=['zip_code'])

# Drop unimportant columns
X = df.drop(columns=['high_stress', 'split', 'date'])

X.head()

,MORTGAGE30US,UNRATE,CPIAUCSL,zhvi_yoy_pct,inventory,days_to_pending,tier_boston_city,tier_inner_suburb,tier_outer_suburb,day,...,zip_code_3868,zip_code_3869,zip_code_3870,zip_code_3871,zip_code_3873,zip_code_3874,zip_code_3878,zip_code_3884,zip_code_3885,zip_code_3887
0,-1.315871,0.857990,-0.692296,1.025253,-0.753537,0.652581,0,0,1,31,...,False,False,False,False,False,False,False,False,False,False
1,-0.949574,4.929049,-0.975248,-0.543465,-0.159508,0.208728,0,0,1,30,...,False,False,False,False,False,False,False,False,False,False
2,-0.128451,-0.402099,-1.198038,-0.164461,0.774547,-0.568015,0,0,1,31,...,False,False,False,False,False,False,False,False,False,False
3,-0.949574,4.929049,-0.975248,0.108575,-0.159508,0.208728,0,0,1,30,...,False,False,False,False,False,False,False,False,False,False
4,1.301457,-0.402099,1.265793,-1.118338,-1.169087,-0.013199,0,1,0,31,...,False,False,False,False,False,False,False,False,False,False


In [16]:
X.shape, y.shape

((22245, 284), (22245, 1))

In [17]:
# Split data into train and test sets
X_train = X[df['split'] == 'train']
X_test = X[df['split'] == 'test']

# Min-max scale the features
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Add bias term
X_train = np.concatenate((X_train, np.ones((X_train.shape[0], 1))), axis=1)
X_test = np.concatenate((X_test, np.ones((X_test.shape[0], 1))), axis=1)

X_train, X_test

(array([[0.01033225, 0.26315789, 0.1927176 , ..., 0.        , 0.        ,
         1.        ],
        [0.12601297, 1.        , 0.09488879, ..., 0.        , 0.        ,
         1.        ],
        [0.38533225, 0.03508772, 0.01786056, ..., 0.        , 0.        ,
         1.        ],
        ...,
        [0.72700567, 0.00877193, 0.74739442, ..., 0.        , 0.        ,
         1.        ],
        [1.        , 0.04385965, 0.85435195, ..., 0.        , 0.        ,
         1.        ],
        [0.85767828, 0.06140351, 0.93296779, ..., 0.        , 0.        ,
         1.        ]], shape=(15571, 285)),
 array([[0.        , 0.28947368, 0.18328017, ..., 0.        , 0.        ,
         1.        ],
        [0.07769449, 0.09649123, 0.39650433, ..., 0.        , 0.        ,
         1.        ],
        [0.03038898, 0.30701754, 0.15790789, ..., 0.        , 0.        ,
         1.        ],
        ...,
        [0.78200972, 0.00877193, 0.76798918, ..., 0.        , 0.        ,
         1.   

In [18]:
# See if there is an imbalance in the classess
print(pd.Series(y_train.flatten()).value_counts())

0    11628
1     3943
Name: count, dtype: int64


In [19]:
# Resample training data to balance classes
X_train_df = pd.DataFrame(X_train)
X_train_df['high_stress'] = y_train.flatten()

# Get low_stress and high_stress classes
low_stress = X_train_df[X_train_df['high_stress'] == 0]
high_stress = X_train_df[X_train_df['high_stress'] == 1]

# Downsample low_stress to match high_stress
low_stress_downsampled = resample(low_stress, replace=False, n_samples=len(high_stress), random_state=42)
balanced = pd.concat([low_stress_downsampled, high_stress]).sample(frac=1, random_state=42)

# Extract X and y from balanced data
X_train = balanced.drop(columns='high_stress').values
y_train = balanced['high_stress'].values.reshape(-1, 1)

In [20]:
pd.DataFrame(X_train).assign(high_stress=y_train.flatten()).to_csv(TRAIN_FILE, index=False)
pd.DataFrame(X_test).assign(high_stress=y_test.flatten()).to_csv(TEST_FILE, index=False)
